# TakaSafe AI-1: LightGBM + Calibration & Conformal Doubt Check for MFS ScamShield\n### End-to-End Pipeline: Transfer (Amount, Receiver, Moment) -> AI-1 Score (LightGBM + Calibration) -> Doubt Check (Conformal Prediction + Novelty Detection)\n\n**Authors**: Md. Sadman Al Islam Shabab (Model Architecture & Explainability Lead) & TakaSafe Research Group  \n**Affiliation**: Model Architecture Team, DIU CPC × upay Hackathon 2026  \n**Objective**: Ultra-reliable, calibrated transaction fraud detection (< 18ms SLA) with distribution-free conformal uncertainty quantification to protect Bangladesh Mobile Financial Service users from social engineering, scams, and pass-through money mules.

## 1. System Architecture Overview\n\n```\n  [ Transfer ]                [ AI-1 Score ]                 [ Doubt Check ]\n (amount, receiver,    -->   (LightGBM + calibration)   -->  (conformal + novelty)  -->  [ ScamShield Intervene / Pass ]\n   moment)                                                (1-alpha=95% coverage)\n```\n\n- **Transfer Node**: Captures high-velocity transaction features: monetary amount (log-scale and deviation ratio relative to 90-day moving average), recipient graph degree (known contact vs first-time recipient vs mule syndicate flag), and moment (Bangladesh Standard Time BST, nocturnal window, 10-minute velocity bursts).\n- **AI-1 Score Node**: Supervised Gradient Boosted Decision Tree (LightGBM) trained on severe class imbalance, calibrated via Isotonic Regression & Platt Scaling so raw tree scores map to true posterior probabilities: $P(\text{Scam} = 1 \mid \hat{p}) \approx \hat{p}$.\n- **Doubt Check Node**: Dual-engine uncertainty quantification:\n  1. **Inductive Split Conformal Prediction**: Produces prediction sets $C(X) \subseteq \{\text{Legitimate}, \text{Scam}\}$ guaranteeing marginal coverage $P(Y \in C(X)) \ge 1 - \alpha$ (at $\alpha = 0.05$). If $|C(X)| = 2$, the model expresses genuine statistical doubt.\n  2. **Novelty Detection (Isolation Forest & Mahalanobis Distance)**: Flags Out-of-Distribution (OOD) transfers to protect against zero-day social engineering and novel fraud patterns.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import IsolationForest
from sklearn.metrics import brier_score_loss, roc_auc_score, f1_score, classification_report
import matplotlib.pyplot as plt
import json

print("Libraries successfully loaded. LightGBM version:", lgb.__version__)

## 2. Dataset Synthesis: Bangladesh MFS Mobile Transactions (10,000 Samples)\nSimulating real-world transaction dynamics across Dhaka, Chittagong, Barishal, and Sylhet corridors.

In [2]:
np.random.seed(2026)
N_SAMPLES = 10_000

# 1. Transfer Amount
observed_avg = np.random.uniform(1500, 3500, N_SAMPLES)
amount_ratio = np.random.exponential(scale=1.2, size=N_SAMPLES)
amount = np.round(observed_avg * amount_ratio, -1)
amount = np.clip(amount, 50, 250_000)

# 2. Receiver Signals
receiver_is_known = np.random.binomial(1, 0.72, N_SAMPLES)
receiver_mule_score = np.where(
    receiver_is_known == 1,
    np.random.beta(1, 10, N_SAMPLES),
    np.random.beta(2, 5, N_SAMPLES)
)
receiver_account_age_days = np.where(
    receiver_is_known == 1,
    np.random.randint(60, 1200, N_SAMPLES),
    np.random.randint(1, 90, N_SAMPLES)
)

# 3. Moment Signals (Bangladesh Standard Time)
moment_hour_bst = np.random.randint(0, 24, N_SAMPLES)
is_nocturnal = np.isin(moment_hour_bst, [0, 1, 2, 3, 4, 5]).astype(int)
velocity_10m_count = np.random.poisson(lam=0.35, size=N_SAMPLES)

# Latent Ground Truth Fraud / Scam Assignment
latent_scam_logit = (
    -3.9
    + 1.8 * np.log10(amount / observed_avg + 0.1)
    + 3.4 * receiver_mule_score
    + 1.3 * (1 - receiver_is_known)
    + 2.1 * is_nocturnal
    + 1.7 * (velocity_10m_count >= 3).astype(int)
)
true_scam_prob = 1 / (1 + np.exp(-latent_scam_logit))
y = np.random.binomial(1, true_scam_prob)

df = pd.DataFrame({
    'amount': amount,
    'observed_avg': observed_avg,
    'amount_ratio': amount / observed_avg,
    'log_amount': np.log10(amount + 1),
    'receiver_is_known': receiver_is_known,
    'receiver_mule_score': receiver_mule_score,
    'receiver_account_age_days': receiver_account_age_days,
    'moment_hour_bst': moment_hour_bst,
    'is_nocturnal': is_nocturnal,
    'velocity_10m_count': velocity_10m_count,
    'is_scam': y
})

print(f"Synthesized Dataset: {df.shape[0]} rows, {df.shape[1]} columns.")
print(f"Positive Scam Prevalence: {df['is_scam'].mean()*100:.2f}%")
df.head()

## 3. Split: Train, Calibration, and Test (60% / 20% / 20%)\nAn independent calibration split is mathematically required for inductive split conformal prediction and isotonic probability calibration.

In [3]:
feature_cols = [
    'amount_ratio', 'log_amount', 'receiver_is_known', 
    'receiver_mule_score', 'receiver_account_age_days', 
    'moment_hour_bst', 'is_nocturnal', 'velocity_10m_count'
]
X = df[feature_cols]
y = df['is_scam']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.40, stratify=y, random_state=42)
X_calib, X_test, y_calib, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(f"Train size:       {len(X_train)} samples ({y_train.sum()} scams)")
print(f"Calibration size: {len(X_calib)} samples ({y_calib.sum()} scams)")
print(f"Test size:        {len(X_test)} samples ({y_test.sum()} scams)")

## 4. Train AI-1 Supervised Model: LightGBM Classifier\nApplying scale_pos_weight to compensate for low scam frequency.

In [4]:
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

base_lgbm = lgb.LGBMClassifier(
    n_estimators=150,
    learning_rate=0.035,
    num_leaves=31,
    max_depth=6,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbose=-1
)
base_lgbm.fit(X_train, y_train)

raw_test_preds = base_lgbm.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, raw_test_preds)
print(f"Base LightGBM ROC-AUC on Test Set: {roc_auc:.4f}")

## 5. Probability Calibration (Isotonic Regression & Platt Scaling)\nGuaranteeing that output probabilities represent calibrated Bayesian posteriors.

In [5]:
calibrated_model = CalibratedClassifierCV(
    estimator=base_lgbm,
    method='isotonic',
    cv='prefit'
)
calibrated_model.fit(X_calib, y_calib)

calibrated_test_preds = calibrated_model.predict_proba(X_test)[:, 1]

brier_raw = brier_score_loss(y_test, raw_test_preds)
brier_cal = brier_score_loss(y_test, calibrated_test_preds)

print(f"Raw LightGBM Brier Score:        {brier_raw:.4f}")
print(f"Calibrated LightGBM Brier Score:   {brier_cal:.4f} (Improved by {(brier_raw - brier_cal)/brier_raw * 100:.1f}%)")

## 6. Doubt Check: Split Conformal Prediction (1 - alpha = 95% Marginal Coverage)\nNon-conformity score: $s_i = 1 - \hat{P}(Y_i \mid X_i)$. We compute the $1-\alpha$ quantile on the calibration set and form prediction sets for incoming transactions.

In [6]:
alpha = 0.05  # 95% coverage guarantee

# Calibration set predicted probabilities
p_calib = calibrated_model.predict_proba(X_calib)

# Non-conformity score: 1 - probability assigned to the true label
non_conformity = 1.0 - p_calib[np.arange(len(y_calib)), y_calib.values]

# Finite sample adjusted quantile
n_cal = len(y_calib)
q_val = np.ceil((n_cal + 1) * (1 - alpha)) / n_cal
q_hat = np.quantile(non_conformity, q_val, method='higher')
print(f"Conformal Quantile Threshold (q_hat at 95% confidence): {q_hat:.4f}")

# Construct prediction sets for Test set
p_test = calibrated_model.predict_proba(X_test)
test_prediction_sets = []
is_doubt_flagged = []

for p0, p1 in p_test:
    pset = []
    if (1.0 - p0) <= q_hat:
        pset.append(0)  # Legitimate
    if (1.0 - p1) <= q_hat:
        pset.append(1)  # Scam
    if len(pset) == 0:
        pset.append(int(p1 >= 0.5))
    test_prediction_sets.append(pset)
    is_doubt_flagged.append(len(pset) > 1)

coverage = np.mean([y_test.iloc[i] in test_prediction_sets[i] for i in range(len(y_test))])
doubt_pct = np.mean(is_doubt_flagged) * 100

print(f"Empirical Coverage on Test Set: {coverage*100:.2f}% (Guaranteed >= { (1-alpha)*100:.0f}%)")
print(f"Doubt Check Trigger Rate:        {doubt_pct:.2f}% of transactions have ambiguous sets {Legitimate, Scam}")

## 7. Doubt Check: Out-Of-Distribution (OOD) Novelty Detection\nDetecting zero-day scam attacks using an Isolation Forest on the (amount, receiver, moment) embedding space.

In [7]:
novelty_features = ['amount_ratio', 'receiver_mule_score', 'is_nocturnal', 'velocity_10m_count']

iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_train[novelty_features])

# Negative score: lower indicates more anomalous/novel
raw_scores = -iso.score_samples(X_test[novelty_features])
novelty_norm = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())
high_novelty = novelty_norm >= 0.62

print(f"Average Novelty Score: {novelty_norm.mean():.3f}")
print(f"Transfers Flagged as High Novelty: {np.sum(high_novelty)} / {len(X_test)}")

## 8. Integrated ScamShield Decision Policy Engine\nPolicy mapping fusing Calibrated AI-1 Score, Conformal Doubt, and Novelty Detection.

In [8]:
# ScamShield Trigger Condition:
# 1) Calibrated Score >= 60 OR
# 2) (Doubt Flagged AND Novelty >= 0.62)
scamshield_hold = (calibrated_test_preds >= 0.60) | (np.array(is_doubt_flagged) & (novelty_norm >= 0.62))

print("ScamShield Trigger Breakdown on 2,000 Test Transfers:")
print(f"Total Interceptions: {np.sum(scamshield_hold)} ({np.mean(scamshield_hold)*100:.2f}%)")
print(f"True Scams Intercepted: {np.sum(scamshield_hold & (y_test == 1))} / {y_test.sum()} (Recall: {np.sum(scamshield_hold & (y_test == 1)) / y_test.sum() * 100:.1f}%)")
print(f"Legitimate Transfers Given 24h Cooling Pause: {np.sum(scamshield_hold & (y_test == 0))} ({np.sum(scamshield_hold & (y_test == 0)) / (y_test == 0).sum() * 100:.2f}% false alarm rate)")

# Export Production Constants
prod_config = {
    "model_name": "TakaSafe-AI-1",
    "architecture": "LightGBM + Calibration + Conformal + Novelty",
    "authors": "Md. Sadman Al Islam Shabab",
    "conformal_q_hat_95": float(q_hat),
    "novelty_threshold": 0.62,
    "calibrated_brier": float(brier_cal),
    "target_coverage": 0.95
}
with open('takasafe_ai1_prod_config.json', 'w') as f:
    json.dump(prod_config, f, indent=2)
print("Successfully saved takasafe_ai1_prod_config.json for frontend pipeline integration!")